In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Path of dataset
zip_path = "/content/drive/MyDrive/Courses/Fall 2025/WESAD.zip"

# Unzip into /content/WESAD
!unzip -q "$zip_path" -d /content/WESAD

!ls /content/WESAD


In [4]:
import os, glob, pickle
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

!pip install lightgbm
from lightgbm import LGBMClassifier


In [5]:
base_path = "/content/WESAD/WESAD"

subject_dirs = sorted(glob.glob(os.path.join(base_path, "S*")))
print("Subjects found:", subject_dirs)

# Inspect first subject
example_dir = subject_dirs[0]
pkl_path = glob.glob(os.path.join(example_dir, "*.pkl"))[0]
print("Example .pkl file:", pkl_path)

with open(pkl_path, 'rb') as f:
    data = pickle.load(f, encoding='latin1')

print("Top-level keys:", data.keys())
print("Chest signal channels:", data["signal"]["chest"].keys())
print("Label array shape:", data["label"].shape)


Subjects found: ['/content/WESAD/WESAD/S10', '/content/WESAD/WESAD/S11', '/content/WESAD/WESAD/S13', '/content/WESAD/WESAD/S14', '/content/WESAD/WESAD/S15', '/content/WESAD/WESAD/S16', '/content/WESAD/WESAD/S17', '/content/WESAD/WESAD/S2', '/content/WESAD/WESAD/S3', '/content/WESAD/WESAD/S4', '/content/WESAD/WESAD/S5', '/content/WESAD/WESAD/S6', '/content/WESAD/WESAD/S7', '/content/WESAD/WESAD/S8', '/content/WESAD/WESAD/S9']
Example .pkl file: /content/WESAD/WESAD/S10/S10.pkl
Top-level keys: dict_keys(['signal', 'label', 'subject'])
Chest signal channels: dict_keys(['ACC', 'ECG', 'EMG', 'EDA', 'Temp', 'Resp'])
Label array shape: (3847200,)


In [6]:
def extract_features_from_subject(signals_dict, labels, subject_id,
                                  fs=700, window_sec=60, step_sec=30):
    min_len = min([len(labels)] + [v.shape[0] for v in signals_dict.values()])
    labels = labels[:min_len]
    signals_dict = {k: v[:min_len] for k,v in signals_dict.items()}

    window_size = window_sec * fs
    step_size = step_sec * fs

    rows = []
    for start in range(0, min_len - window_size + 1, step_size):
        end = start + window_size

        labels_win = labels[start:end]
        uniq, counts = np.unique(labels_win, return_counts=True)
        label_mode = uniq[counts.argmax()]

        # Map WESAD labels → binary stress
        if label_mode == 2:
            target = 1     # stress
        elif label_mode in [1, 3, 4]:
            target = 0     # non-stress
        else:
            continue       # skip 0,5,6,7

        feats = {}
        for name, data in signals_dict.items():
            win = data[start:end]

            if win.ndim == 1:
                feats[f"{name}_mean"] = float(win.mean())
                feats[f"{name}_std"] = float(win.std())
                feats[f"{name}_min"] = float(win.min())
                feats[f"{name}_max"] = float(win.max())

            elif win.ndim == 2:  # e.g., ACC 3-axis
                for ch in range(win.shape[1]):
                    wch = win[:, ch]
                    feats[f"{name}_ch{ch}_mean"] = float(wch.mean())
                    feats[f"{name}_ch{ch}_std"]  = float(wch.std())
                    feats[f"{name}_ch{ch}_min"]  = float(wch.min())
                    feats[f"{name}_ch{ch}_max"]  = float(wch.max())

        feats["subject_id"] = subject_id
        feats["label"] = target
        rows.append(feats)

    return rows


In [7]:
all_rows = []

for s_dir in subject_dirs:
    sid = int(os.path.basename(s_dir).replace("S",""))
    pkl_path = glob.glob(os.path.join(s_dir, "*.pkl"))[0]

    print(f"Processing subject {sid} ...")
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')

    chest = data['signal']['chest']
    chest_signals = {
        "ECG":  chest["ECG"],
        "EDA":  chest["EDA"],
        "EMG":  chest["EMG"],
        "Resp": chest["Resp"],
        "Temp": chest["Temp"],
        "ACC":  chest["ACC"],
    }

    labels = data["label"].flatten()

    rows = extract_features_from_subject(
        chest_signals,
        labels,
        subject_id=sid,
        fs=700,
        window_sec=60,
        step_sec=30
    )
    all_rows.extend(rows)

df = pd.DataFrame(all_rows)
print(df.shape)
df.head()


Processing subject 10 ...
Processing subject 11 ...
Processing subject 13 ...
Processing subject 14 ...
Processing subject 15 ...
Processing subject 16 ...
Processing subject 17 ...
Processing subject 2 ...
Processing subject 3 ...
Processing subject 4 ...
Processing subject 5 ...
Processing subject 6 ...
Processing subject 7 ...
Processing subject 8 ...
Processing subject 9 ...
(1499, 34)


,ECG_ch0_mean,ECG_ch0_std,ECG_ch0_min,ECG_ch0_max,EDA_ch0_mean,EDA_ch0_std,EDA_ch0_min,EDA_ch0_max,EMG_ch0_mean,EMG_ch0_std,...,ACC_ch1_mean,ACC_ch1_std,ACC_ch1_min,ACC_ch1_max,ACC_ch2_mean,ACC_ch2_std,ACC_ch2_min,ACC_ch2_max,subject_id,label
0,0.001702,0.136454,-0.667923,0.821457,0.739102,0.013610,0.563812,0.863266,-0.002243,0.009838,...,0.082801,0.014365,-0.0792,0.1432,-0.205440,0.013360,-0.2962,-0.1076,10,0
1,0.001372,0.151259,-0.667923,0.838211,0.744245,0.013809,0.594711,0.881195,-0.002273,0.009424,...,0.064452,0.025789,-0.0792,0.1846,-0.264077,0.059623,-0.3762,-0.1076,10,0
2,0.000920,0.159931,-0.664948,0.871902,0.747692,0.013591,0.453186,0.881195,-0.002285,0.009043,...,0.045407,0.014106,-0.0702,0.1846,-0.315126,0.017693,-0.3762,-0.1394,10,0
3,0.001438,0.159889,-0.700928,0.871902,0.749359,0.013476,0.453186,0.851440,-0.002297,0.009059,...,0.044194,0.007487,0.0244,0.1018,-0.311683,0.011957,-0.3594,-0.2706,10,0
4,0.001176,0.160889,-0.700928,0.827225,0.751742,0.013415,0.595474,0.860977,-0.002275,0.009096,...,0.046866,0.006534,0.0254,0.0842,-0.318932,0.011402,-0.3662,-0.2754,10,0


In [8]:
X = df.drop(columns=["label", "subject_id"])
y = df["label"].values
groups = df["subject_id"].values

gkf = GroupKFold(n_splits=len(np.unique(groups)))

y_true_all = []
y_prob_all = []

for train_idx, test_idx in gkf.split(X, y, groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:,1]

    y_true_all.extend(y_test)
    y_prob_all.extend(y_prob)

y_true_all = np.array(y_true_all)
y_prob_all = np.array(y_prob_all)
y_pred_all = (y_prob_all >= 0.5).astype(int)

print("LightGBM LOSO Accuracy :", accuracy_score(y_true_all, y_pred_all))
print("LightGBM LOSO F1       :", f1_score(y_true_all, y_pred_all))
print("LightGBM LOSO AUC      :", roc_auc_score(y_true_all, y_prob_all))
print("\nClassification Report:\n")
print(classification_report(y_true_all, y_pred_all))


[LightGBM] [Info] Number of positive: 308, number of negative: 1088
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000846 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8020
[LightGBM] [Info] Number of data points in the train set: 1396, number of used features: 32
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.220630 -> initscore=-1.261997
[LightGBM] [Info] Start training from score -1.261997
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

In [9]:
final_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
final_model.fit(X, y)


[LightGBM] [Info] Number of positive: 332, number of negative: 1167
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000805 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8121
[LightGBM] [Info] Number of data points in the train set: 1499, number of used features: 32
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.221481 -> initscore=-1.257057
[LightGBM] [Info] Start training from score -1.257057
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

LGBMClassifier(colsample_bytree=0.8, learning_rate=0.05, n_estimators=300,
               random_state=42, subsample=0.8)

In [10]:
def summarize_stress(stress_probs, last_n=10):
    if len(stress_probs) == 0:
        return None

    last_n = min(last_n, len(stress_probs))
    recent = stress_probs[-last_n:]

    return {
        "avg": float(np.mean(recent)),
        "max": float(np.max(recent)),
        "n": last_n
    }

In [17]:
# Test without LLM

def ai_recommendation(summary):
    if summary is None:
        return "No stress data available yet."

    avg_s, max_s, n = summary["avg"], summary["max"], summary["n"]

    if avg_s > 0.8:
        return (f"In the last ~{n} windows, stress has been HIGH (avg {avg_s:.2f}).\n"
                "Recommendation: Take a 10–15 min break, deep breathing (4–6), "
                "or step away from your task.")

    if avg_s > 0.5 or max_s > 0.8:
        return (f"Recent stress is MODERATE (avg {avg_s:.2f}).\n"
                "Recommendation: 5 min walk, stretching, light breathing exercise.")

    return (f"Stress is LOW (avg {avg_s:.2f}).\n"
            "Recommendation: Keep steady; note habits that keep you calm.")


In [20]:
demo_subject = 8
mask = (df["subject_id"] == demo_subject)
X_demo = df.loc[mask].drop(columns=["label", "subject_id"])

probs_demo = final_model.predict_proba(X_demo)[:,1]
summary_demo = summarize_stress(probs_demo, last_n=10)
msg = ai_recommendation(summary_demo)

print("AI Recommendation:\n")
print(msg)

AI Recommendation:

Stress is LOW (avg 0.00).
Recommendation: Keep steady; note habits that keep you calm.


In [14]:
# Install Gradio
!pip install gradio

In [15]:
import gradio as gr
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import google.generativeai as genai

model = pickle.load(open("lightgbm_final.pkl", "rb"))
FEATURE_COLUMNS = model.feature_name_
SESSION_MEMORY = []

def analyze(file, api_key):
    df = pd.read_csv(file.name)
    df = df[FEATURE_COLUMNS]
    prob = float(model.predict_proba(df)[0][1])
    SESSION_MEMORY.append(prob)
    if len(SESSION_MEMORY) > 10:
        SESSION_MEMORY.pop(0)

    genai.configure(api_key=api_key)
    gm = genai.GenerativeModel("gemini-1.5-flash")

    avg_s = float(np.mean(SESSION_MEMORY))
    max_s = float(np.max(SESSION_MEMORY))
    n = len(SESSION_MEMORY)

    prompt = f"""
    Latest stress probability: {prob:.2f}
    Average: {avg_s:.2f}
    Maximum observed: {max_s:.2f}

    Provide a warm, concise interpretation and 3 personalized recommendations.
    """

    ai_text = gm.generate_content(prompt).text

    fig, ax = plt.subplots(figsize=(5.2,3.2))
    sns.lineplot(x=list(range(1,n+1)), y=SESSION_MEMORY, marker="o", ax=ax, linewidth=2)
    ax.set_ylim(0,1)
    ax.set_title("Stress Trend")
    ax.set_xlabel("Window")
    ax.set_ylabel("Probability")
    fig.tight_layout()

    return f"{prob:.3f}", ai_text, fig


CSS = """
body {background:#f5f6fa;}
.gradio-container {font-family:'Inter', sans-serif;}
.card {
    background:white;
    padding:24px;
    border-radius:14px;
    box-shadow:0 2px 10px rgba(0,0,0,0.05);
    margin-bottom:16px;
}
.big-title {
    font-size:34px;
    font-weight:800;
    margin-bottom:6px;
}
.subtitle {
    font-size:15px;
    color:#666;
    margin-bottom:20px;
}
.metric {
    font-size:30px;
    font-weight:700;
    color:#E86E24;
}
.footer {
    margin-top:40px;
    text-align:center;
    font-size:14px;
    color:#888;
}
button.primary {
    background:#FF7A00 !important;
    color:white !important;
    border:none !important;
}
"""

with gr.Blocks(css=CSS, title="AILANDER — Stress Detection AI") as demo:

    gr.HTML("""
        <div class='big-title'>🧠 Stress Detection AI System</div>
        <div class='subtitle'>Powered by LightGBM + Gemini AI — AILANDER Project</div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.HTML("<div class='card'>")
            file_input = gr.File(label="Upload Sensor CSV")
            api_key_input = gr.Textbox(label="Gemini API Key", type="password")
            btn = gr.Button("Analyze", elem_classes="primary")
            gr.HTML("</div>")

        with gr.Column(scale=2):
            gr.HTML("<div class='card'>")
            stress_out = gr.Text(label="Stress Probability", elem_classes="metric")
            ai_out = gr.Text(label="AI Recommendation", lines=10)
            gr.HTML("</div>")

    gr.HTML("<div class='card'>")
    trend_plot = gr.Plot()
    gr.HTML("</div>")

    gr.HTML("<div class='footer'>Developed by Team AILANDER — Fall 2025</div>")

    btn.click(
        analyze,
        inputs=[file_input, api_key_input],
        outputs=[stress_out, ai_out, trend_plot]
    )

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://06ca5a503345a118d0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Without API_Demo Data

In [16]:
import gradio as gr
import pandas as pd
import numpy as np
import pickle
import os
import matplotlib.pyplot as plt
import seaborn as sns
import google.generativeai as genai

# ==============================
# Load model if present, else dummy
# ==============================
if os.path.exists("lightgbm_final.pkl"):
    model = pickle.load(open("lightgbm_final.pkl", "rb"))
    FEATURE_COLUMNS = model.feature_name_
else:
    model = None
    FEATURE_COLUMNS = None

SESSION_MEMORY = []

# ==============================
# AI RECOMMENDATION (No LLM fallback)
# ==============================
def fallback_ai(prob, avg_s, max_s):
    if avg_s > 0.8 or prob > 0.8:
        return (
            f"Stress appears HIGH (prob {prob:.2f}).\n"
            "Recommendation: Take a 10–15 min break, hydrate, and try deep breathing."
        )
    if avg_s > 0.5:
        return (
            f"Stress appears MODERATE (prob {prob:.2f}).\n"
            "Recommendation: Stretch, walk briefly, and adjust posture."
        )
    return (
        f"Stress appears LOW (prob {prob:.2f}).\n"
        "Recommendation: Keep your routine; maintain healthy habits."
    )

# ==============================
# Main Analyze Function
# ==============================
def analyze(file, api_key):

    # -----------------------------
    # 1. Compute prediction (real or dummy)
    # -----------------------------
    if model is not None and file is not None:
        df = pd.read_csv(file.name)
        df = df[FEATURE_COLUMNS]
        prob = float(model.predict_proba(df)[0][1])
    else:
        prob = float(np.random.uniform(0.1, 0.9))  # dummy probability

    SESSION_MEMORY.append(prob)
    if len(SESSION_MEMORY) > 10:
        SESSION_MEMORY.pop(0)

    avg_s = float(np.mean(SESSION_MEMORY))
    max_s = float(np.max(SESSION_MEMORY))
    n = len(SESSION_MEMORY)

    # -----------------------------
    # 2. Use Gemini only if API key is valid
    # -----------------------------
    if api_key.strip() == "":
        ai_text = fallback_ai(prob, avg_s, max_s)
    else:
        try:
            genai.configure(api_key=api_key)
            gm = genai.GenerativeModel("gemini-1.5-flash")

            prompt = f"""
            Stress probability: {prob:.2f}
            Average: {avg_s:.2f}
            Maximum: {max_s:.2f}

            Give a warm, concise interpretation & 3 personalized recommendations.
            """

            ai_text = gm.generate_content(prompt).text
        except:
            ai_text = fallback_ai(prob, avg_s, max_s)

    # -----------------------------
    # 3. Build Plot
    # -----------------------------
    fig, ax = plt.subplots(figsize=(5.2, 3.2))
    sns.lineplot(x=list(range(1, n + 1)), y=SESSION_MEMORY, marker="o", ax=ax, linewidth=2)
    ax.set_ylim(0, 1)
    ax.set_title("Stress Trend")
    ax.set_xlabel("Window")
    ax.set_ylabel("Probability")
    fig.tight_layout()

    return f"{prob:.3f}", ai_text, fig


# ==============================
# UI Styling
# ==============================
CSS = """
body {background:#f5f6fa;}
.gradio-container {font-family:'Inter', sans-serif;}
.card {
    background:white;
    padding:24px;
    border-radius:14px;
    box-shadow:0 2px 10px rgba(0,0,0,0.05);
    margin-bottom:16px;
}
.big-title {
    font-size:34px;
    font-weight:800;
    margin-bottom:6px;
}
.subtitle {
    font-size:14px;
    color:#777;
    margin-bottom:18px;
}
.metric {
    font-size:30px;
    font-weight:700;
    color:#E86E24;
}
.footer {
    margin-top:40px;
    text-align:center;
    font-size:14px;
    color:#888;
}
button.primary {
    background:#FF7A00 !important;
    color:white !important;
    border:none !important;
}
"""

# ==============================
# GRADIO UI
# ==============================
with gr.Blocks(css=CSS, title="AILANDER — Stress Detection AI") as demo:

    gr.HTML("""
        <div class='big-title'>🧠 Stress Detection AI System</div>
        <div class='subtitle'>A lightweight demo. Real-time AI responses appear when the API key is available.</div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.HTML("<div class='card'>")
            file_input = gr.File(label="Upload Sensor CSV")
            api_key_input = gr.Textbox(label="Gemini API Key (optional)", type="password")
            btn = gr.Button("Analyze", elem_classes="primary")
            gr.HTML("</div>")

        with gr.Column(scale=2):
            gr.HTML("<div class='card'>")
            stress_out = gr.Text(label="Stress Probability", elem_classes="metric")
            ai_out = gr.Text(label="AI Recommendation", lines=10)
            gr.HTML("</div>")

    gr.HTML("<div class='card'>")
    trend_plot = gr.Plot()
    gr.HTML("</div>")

    gr.HTML("<div class='footer'>Developed by Team AILANDER — Fall 2025</div>")

    btn.click(analyze, [file_input, api_key_input], [stress_out, ai_out, trend_plot])

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://21606f9a4c962c32fc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
